# 第5章 行情数据获取

> 本 notebook 为《量化投资》配套代码，可在 Jupyter / Colab / Binder 中运行。

In [ ]:
# Colab 自举：在 Colab 上自动克隆仓库并安装 qfin（本地用 uv 运行会自动跳过）
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('qfin') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','-q','https://github.com/albertandking/quant-investing.git'], check=False)
        os.chdir('quant-investing')
        subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=False)
        subprocess.run([sys.executable,'scripts/make_sample_data.py'], check=False)


## 离线：用内置数据完成加载与清洗（断网可跑）

In [ ]:
import pandas as pd
from qfin import load_prices, load_close

panel = load_prices()
print(panel.head())
print(panel['symbol'].nunique(), '只股票')

In [ ]:
one = panel[panel['symbol'] == 'STK001'].sort_values('date').set_index('date').copy()
ohlcv = ['open','high','low','close','volume']
one[ohlcv] = one[ohlcv].apply(pd.to_numeric, errors='coerce')
one['close'] = one['close'].ffill()
print(one.tail())

In [ ]:
close = load_close()
print('收盘价宽表形状：', close.shape)
print(close.iloc[-3:, :4])

## 联网（可选）：用 AkShare 抓真实行情
需要 `uv sync --extra data`，且需联网。未安装/无网时自动跳过。

In [ ]:
try:
    import akshare as ak
    df = ak.stock_zh_a_hist(symbol='600519', period='daily',
                            start_date='20230101', end_date='20231231', adjust='qfq')
    print(df.head())
except Exception as e:
    print('跳过联网示例（未安装 akshare 或无网络）：', e)